In [ ]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 19.8 MB/s eta 0:00:00


## AgriMetrics: Broiler Batch Analytics — Synthetic Data Generation

This notebook generates a synthetic dataset simulating **12 broiler production batches**
for a fictional operation, *Origin Fields Poultry Farms Ltd.* (Ibadan, Nigeria).

**What it generates:**
- `batches.csv` — batch metadata (breed, stock size, start/end dates)
- `daily_logs.csv` — daily mortality, feed consumption, and weight records across each batch's 42-day cycle
- `financials.csv` — expense entries per batch (feed, medication, labor, utilities, misc.)
- `batch_summary_metrics.csv` — per-batch FCR and mortality rate, aggregated from the daily logs

**Key design choice — FCR emerges, it isn't set directly:**
Feed Conversion Ratio (FCR) is not hardcoded. It's the natural result of dividing total feed
consumed by final weight, both of which come from a **breed-specific logistic growth curve**
(different target weights for Cobb 500, Ross 308, and Hubbard Classic). This mirrors how FCR
actually works on a real farm — it's an outcome of growth and feed intake, not an input.

**Reproducibility:**
`SEED = 42` is applied to `Faker`, `NumPy`, and `random` so this notebook produces the exact
same dataset every time it's run — which is what allows the SQL, Excel, and Power BI layers
of this project to reference consistent, traceable numbers throughout.

In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

# Initialize Faker
fake = Faker()
SEED = 42
Faker.seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# ==========================================
# 1. GENERATE BATCHES DATA
# ==========================================
num_batches = 12
batch_ids = [f"BATCH-{2026}-{str(i).zfill(3)}" for i in range(1, num_batches + 1)]
feed_types = ["Starter Crumbles", "Grower Pellets", "Finisher Pellets"]

# Breed-specific performance targets (industry-standard 42-day broiler benchmarks).
# target_weight_kg = realistic live weight at market age.
# growth_midpoint / growth_rate = shape of the logistic growth curve.
# FCR is NOT set directly here — it emerges naturally from weight vs. feed intake,
# same as it would on a real farm.
BREED_PARAMS = {
    "Cobb 500":        {"target_weight_kg": 2.60, "growth_midpoint": 23, "growth_rate": 0.145},
    "Ross 308":        {"target_weight_kg": 2.50, "growth_midpoint": 23, "growth_rate": 0.140},
    "Hubbard Classic": {"target_weight_kg": 2.30, "growth_midpoint": 24, "growth_rate": 0.130},
}
breed_types = list(BREED_PARAMS.keys())

start_dates = [datetime(2026, 1, 15) + timedelta(days=i * 20) for i in range(num_batches)]

batches_data = []
for i in range(num_batches):
    b_id = batch_ids[i]
    start_date = start_dates[i]
    duration_days = 42  # standard 6-week broiler cycle
    end_date = start_date + timedelta(days=duration_days)
    initial_stock = random.randint(1000, 5000)

    batches_data.append({
        "batch_id": b_id,
        "start_date": start_date.strftime("%Y-%m-%d"),
        "end_date": end_date.strftime("%Y-%m-%d"),
        "initial_stock": initial_stock,
        "breed_type": random.choice(breed_types),
        "primary_feed": random.choice(feed_types),
        "status": "Completed" if end_date < datetime.now() else "Active"
    })

batches_df = pd.DataFrame(batches_data)

# ==========================================
# 2. GENERATE DAILY OPERATIONAL LOGS
# ==========================================
daily_logs = []

for _, batch in batches_df.iterrows():
    b_id = batch["batch_id"]
    breed = batch["breed_type"]
    params = BREED_PARAMS[breed]
    start_date = datetime.strptime(batch["start_date"], "%Y-%m-%d")
    initial_stock = batch["initial_stock"]

    # Per-batch random variance so batches of the same breed don't all
    # converge on an identical FCR (real farms never do).
    weight_variance = random.uniform(0.95, 1.05)
    feed_variance = random.uniform(0.96, 1.06)
    target_weight_g = params["target_weight_kg"] * 1000 * weight_variance

    current_stock = initial_stock
    for day in range(1, 43):  # 42-day cycle
        log_date = start_date + timedelta(days=day - 1)

        # Mortality: baseline daily risk, with occasional heat-stress /
        # minor disease spikes layered on top
        mortality_prob = random.choices([0, 1, 2, 3], weights=[0.6, 0.25, 0.1, 0.05])[0]
        is_stress_event = random.random() < 0.05  # ~5% of days
        if is_stress_event:
            mortality_prob += random.randint(2, 6)

        mortality = min(current_stock, mortality_prob)
        current_stock -= mortality

        # Weight gain: breed-specific logistic growth curve, capped at a
        # realistic market weight instead of a fixed generic asymptote
        base_g = 40  # day-old chick weight
        avg_weight_g = base_g + (target_weight_g - base_g) / (
            1 + np.exp(-params["growth_rate"] * (day - params["growth_midpoint"]))
        )
        if is_stress_event:
            avg_weight_g *= random.uniform(0.97, 0.99)  # stress slows growth slightly
        total_weight_kg = (current_stock * avg_weight_g) / 1000

        # Feed consumption: unchanged intake curve (this part was already
        # realistic — roughly 4kg/bird cumulative over 42 days)
        feed_per_bird_g = (20 + (day * 3.5)) * feed_variance
        if is_stress_event:
            feed_per_bird_g *= random.uniform(1.03, 1.08)  # eat more, convert worse
        feed_consumed_kg = round((current_stock * feed_per_bird_g) / 1000, 2)

        daily_logs.append({
            "log_date": log_date.strftime("%Y-%m-%d"),
            "batch_id": b_id,
            "age_days": day,
            "surviving_stock": current_stock,
            "mortality_count": mortality,
            "feed_consumed_kg": feed_consumed_kg,
            "total_weight_kg": round(total_weight_kg, 2)
        })

daily_logs_df = pd.DataFrame(daily_logs)

# ==========================================
# 3. GENERATE FINANCIAL EXPENSES DATA
# ==========================================
expense_categories = ["Feed Purchase", "Medication & Vaccines", "Labor", "Utility (Power/Water)", "Miscellaneous"]
financials_data = []

for _, batch in batches_df.iterrows():
    b_id = batch["batch_id"]
    start_date = datetime.strptime(batch["start_date"], "%Y-%m-%d")
    initial_stock = batch["initial_stock"]

    # Generate 4-6 expense entries per batch lifecycle
    for _ in range(random.randint(4, 6)):
        category = random.choice(expense_categories)

        if category == "Feed Purchase":
            cost = initial_stock * random.uniform(1200, 1500)
        elif category == "Medication & Vaccines":
            cost = initial_stock * random.uniform(150, 250)
        elif category == "Labor":
            cost = random.randint(50000, 150000)
        elif category == "Utility (Power/Water)":
            cost = random.randint(30000, 80000)
        else:
            cost = random.randint(10000, 40000)

        expense_date = start_date + timedelta(days=random.randint(0, 40))

        financials_data.append({
            "expense_date": expense_date.strftime("%Y-%m-%d"),
            "batch_id": b_id,
            "expense_category": category,
            "cost_naira": round(cost, 2)
        })

financials_df = pd.DataFrame(financials_data)

# ==========================================
# 4. CALCULATE AGRI-METRICS & EXPORT
# ==========================================
batch_summary = daily_logs_df.groupby("batch_id").agg(
    total_mortality=("mortality_count", "sum"),
    total_feed_consumed_kg=("feed_consumed_kg", "sum"),
    final_weight_kg=("total_weight_kg", "last")
).reset_index()

batch_summary = batch_summary.merge(batches_df, on="batch_id")
batch_summary["mortality_rate_pct"] = round((batch_summary["total_mortality"] / batch_summary["initial_stock"]) * 100, 2)
batch_summary["fcr"] = round(batch_summary["total_feed_consumed_kg"] / batch_summary["final_weight_kg"], 2)

print("--- Data Generation Complete ---")
print(f"Generated {len(batches_df)} Batches")
print(f"Generated {len(daily_logs_df)} Daily Log Rows")
print(f"Generated {len(financials_df)} Financial Expense Entries")
print("\nSample Batch Performance Summary:")
print(batch_summary[["batch_id", "breed_type", "initial_stock", "mortality_rate_pct", "fcr"]].head(12))

# Export files to CSV for local download
batches_df.to_csv("batches.csv", index=False)
daily_logs_df.to_csv("daily_logs.csv", index=False)
financials_df.to_csv("financials.csv", index=False)
batch_summary.to_csv("batch_summary_metrics.csv", index=False)

--- Data Generation Complete ---
Generated 12 Batches
Generated 504 Daily Log Rows
Generated 60 Financial Expense Entries

Sample Batch Performance Summary:
          batch_id       breed_type  initial_stock  mortality_rate_pct   fcr
0   BATCH-2026-001         Cobb 500           3619                0.86  1.53
1   BATCH-2026-002         Ross 308           4037                0.87  1.71
2   BATCH-2026-003         Cobb 500           1914                1.72  1.58
3   BATCH-2026-004  Hubbard Classic           1419                4.16  1.95
4   BATCH-2026-005  Hubbard Classic           4654                0.58  2.00
5   BATCH-2026-006         Ross 308           3418                1.20  1.90
6   BATCH-2026-007         Cobb 500           1122                3.30  1.73
7   BATCH-2026-008  Hubbard Classic           1952                1.84  2.01
8   BATCH-2026-009  Hubbard Classic           1108                2.89  1.93
9   BATCH-2026-010  Hubbard Classic           3932                0.64  1